In [13]:
import torch
from torch import nn
import numpy as np
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint


In [49]:
class PatchEmbed(nn.Module):
    def __init__(self, patch_size=4, in_c=3, embed_dim=96, norm_layer=None):
        super().__init__()
        patch_size = (patch_size, patch_size)
        self.patch_size = patch_size
        self.in_chans = in_c
        self.embed_dim = embed_dim
        self.proj = nn.Conv2d(in_c, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()

    def forward(self, x):
        _, _, H, W = x.shape

        # padding
        # 如果输入图片的H，W不是patch_size的整数倍，需要进行padding
        pad_input = (H % self.patch_size[0] != 0) or (W % self.patch_size[1] != 0)
        if pad_input:
            # to pad the last 3 dimensions,从最后一个维度向前指定
            # (W_left, W_right, H_top,H_bottom, C_front, C_back)
            x = F.pad(x, (0, self.patch_size[1] - W % self.patch_size[1],
                          0, self.patch_size[0] - H % self.patch_size[0],
                          0, 0))

        # 下采样patch_size倍
        x = self.proj(x)
        _, _, H, W = x.shape
        # flatten: [B, C, H, W] -> [B, C, HW]
        # transpose: [B, C, HW] -> [B, HW, C]
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x, H, W

class PatchMerging(nn.Module):
    def __init__(self,dim,norm_layer=nn.LayerNorm):
        super().__init__()
        self.norm=norm_layer(4*dim)
        self.reduction=nn.Linear(4*dim,2*dim,bias=False)


    def forward(self,x,H,W):
        """

        :param x: [B,H*W,C]
        :return:
        """
        B,N,C=x.shape
        assert N == H * W, "input feature has wrong size"

        x = x.view(B, H, W, C)

        # padding
        # 如果输入feature map的H，W不是2的整数倍，需要进行padding
        pad_input = (H % 2 == 1) or (W % 2 == 1)
        if pad_input:
            # to pad the last 3 dimensions, starting from the last dimension and moving forward.
            # (C_front, C_back, W_left, W_right, H_top, H_bottom)
            # 注意这里的Tensor通道是[B, H, W, C]，所以会和官方文档有些不同
            x = F.pad(x, (0, 0, 0, W % 2, 0, H % 2))

        x0 = x[:, 0::2, 0::2, :]  # [B, H/2, W/2, C]
        x1 = x[:, 1::2, 0::2, :]  # [B, H/2, W/2, C]
        x2 = x[:, 0::2, 1::2, :]  # [B, H/2, W/2, C]
        x3 = x[:, 1::2, 1::2, :]  # [B, H/2, W/2, C]
        x = torch.cat([x0, x1, x2, x3], -1)  # [B, H/2, W/2, 4*C]
        x = x.view(B, -1, 4 * C)  # [B, H/2*W/2, 4*C]

        x = self.norm(x)
        x = self.reduction(x)  # [B, H/2*W/2, 2*C]

        return x

In [41]:
def window_partition(x,window_size):
        B,N,patch_dim=x.shape
        img_size=int(N**0.5)
        patch_embed=x.view(B,img_size,img_size,patch_dim)
        window_patch=patch_embed.view(B,img_size//window_size,window_size,
                                      img_size//window_size,window_size,
                                      patch_dim).permute(0,1,3,2,4,5).contiguous()
        window_patch=window_patch.view(-1,window_size*window_size,patch_dim)
        return window_patch

def window_recover(x,win_size,H,W):
        _,N,patch_dim=x.shape

        window_patch=x.view(-1,H//win_size,W//win_size,win_size,win_size,patch_dim)
        window_patch=window_patch.permute(0,1,3,2,4,5).contiguous().view(-1,H,W,patch_dim).view(-1,H*W,patch_dim)
        return window_patch

In [43]:
class WindowAttention(nn.Module):
    def __init__(self,embed_size,head_num,attn_drop,drop_ratio,win_size,qkv_bias=False):
        super().__init__()
        self.embed_size=embed_size
        self.head_num=head_num
        self.head_dim=embed_size//head_num
        self.attn_drop=nn.Dropout(p=attn_drop)
        self.drop=nn.Dropout(p=drop_ratio)
        self.qkv=nn.Linear(embed_size,3*embed_size,bias=qkv_bias)
        self.dropout=nn.Dropout(p=drop_ratio)
        self.scale=self.head_dim**-0.5
        self.proj=nn.Linear(self.embed_size,self.embed_size)
        self.win_size=win_size
        self.softmax=nn.Softmax(dim=-1)

        # define a parameter table of relative position bias
        self.relative_position_bias = nn.Parameter(
            torch.zeros((2 * win_size - 1) * (2 * win_size - 1), head_num))  # [2*Mh-1 * 2*Mw-1, nH]

        # get pair-wise relative position index for each token inside the window
        coords_h = torch.arange(self.win_size)
        coords_w = torch.arange(self.win_size)
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing="ij"))  # [2, Mh, Mw]
        coords_flatten = torch.flatten(coords, 1)  # [2, Mh*Mw]
        # [2, Mh*Mw, 1] - [2, 1, Mh*Mw]
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # [2, Mh*Mw, Mh*Mw]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # [Mh*Mw, Mh*Mw, 2]
        relative_coords[:, :, 0] += self.win_size - 1  # shift to start from 0
        relative_coords[:, :, 1] += self.win_size - 1
        relative_coords[:, :, 0] *= 2 * self.win_size - 1
        relative_position_index = relative_coords.sum(-1)  # [Mh*Mw, Mh*Mw]
        self.register_buffer("relative_position_index", relative_position_index)

    def forward(self,x,mask=None):
        #[B,N,C]
        B,N,C=x.shape
        qkv=self.qkv(x).reshape(B,N,3,self.head_num,C//self.head_num).permute(2,0,3,1,4)
        #[B,head_num,N,head_dim]
        q,k,v=qkv.unbind(0)
        attn=q@k.transpose(-1,-2)*self.scale
        relative_position_bias=self.relative_position_bias[self.relative_position_index].view(
            self.win_size*self.win_size,self.win_size*self.win_size,-1
        )
        relative_position_bias=relative_position_bias.permute(2,0,1)
        attn+=relative_position_bias.unsqueeze(0)
        if mask is not None:
            #mask: [nW,wh*ww,wh*ww]
            nW=mask.shape[0]
            attn=attn.reshape(B//nW,nW,self.head_num,N,N)+mask[None,:,None,:,:]
            attn=attn.reshape(B,self.head_num,N,N)
            attn=self.softmax(attn)
        else:
            attn=self.softmax(attn)

        attn=self.attn_drop(attn)
        x=attn@v
        x=x.permute(0,2,1,3).reshape(B,N,C)
        x=self.drop(self.proj(x))
        return x

In [17]:
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.):
        super().__init__()
        self.drop_prob=drop_prob

    def forward(self,x):
        if not self.training:
            return x

        keep_prob=1-self.drop_prob
        shape=(x.shape[0],)+(1,)*(x.ndim-1)
        random_tensor=torch.randn(shape,dtype=x.dtype,device=x.device)+keep_prob
        keep_tensor=random_tensor.floor_()
        x=x.div(keep_prob)*keep_tensor
        return x


In [45]:
class Block(nn.Module):
    def __init__(self,embed_dim,head_num,win_size,attn_drop,drop,
                 drop_path,shift_size=0,mlp_ratio=4,
                 act_layer=nn.GELU,norm_layer=nn.LayerNorm,
                 qkv_bias=False):
        super().__init__()
        if shift_size>0:
            self.shift_size=shift_size
        else:
            self.shift_size=0

        self.norm_layer=norm_layer
        self.drop_path=DropPath(drop_path) if drop_path is not None else nn.Identity()
        self.drop=nn.Dropout(p=drop)
        self.act=act_layer
        self.attn_drop=attn_drop
        self.win_size=win_size
        self.mlp=nn.Sequential(
            nn.Linear(embed_dim,mlp_ratio*embed_dim),
            nn.Dropout(p=drop),
            act_layer(),
            nn.Linear(mlp_ratio*embed_dim,embed_dim),
            nn.Dropout(p=drop),
        )

        self.norm1=norm_layer(embed_dim)
        self.norm2=norm_layer(embed_dim)
        self.window_attn=WindowAttention(embed_dim,head_num,attn_drop,drop,win_size,qkv_bias)

    def _create_mask(self,x,H,W):
        device=x.device
        Hp=int(np.ceil(H/self.win_size))*self.win_size
        Wp=int(np.ceil(W/self.win_size))*self.win_size
        mask=torch.zeros((1,Hp,Wp,1),device=device)
        h_slice=(slice(0,-self.win_size),slice(-self.win_size,-self.shift_size),slice(-self.shift_size,0))
        w_slice=(slice(0,-self.win_size),slice(-self.win_size,-self.shift_size),slice(-self.shift_size,0))

        index=0
        for h in h_slice:
            for w in w_slice:
                mask[:,h,w,:]=index
                index+=1
        #[1*nW,win_size*win_size,1]
        mask_windows=window_partition(mask.reshape(1,-1,1),self.win_size)
        mask_windows=mask_windows.reshape(-1,self.win_size*self.win_size)
        #[nW,1,win_size*win_size]-[nW,win_size*win_size,1]
        attn_mask=mask_windows.unsqueeze(1)-mask_windows.unsqueeze(2)
        return attn_mask

    def forward(self,x):
        H,W=self.H,self.W
        B,N,C=x.shape
        shortcut=x
        x=self.norm1(x)
        x=x.reshape(-1,H,W,C)
        pad_l=pad_t=0
        pad_r=(self.win_size-W%self.win_size)%self.win_size
        pad_b=(self.win_size-H%self.win_size)%self.win_size
        x=F.pad(x,(0,0,pad_l,pad_r,pad_t,pad_b))
        _,Hp,Wp,_=x.shape

        mask=None
        if self.shift_size>0:
            mask=self._create_mask(x,Hp,Wp)
            shift_x=torch.roll(x,(-self.win_size//2,-self.win_size//2),(1,2))
        else:
            shift_x=x
        #partition
        #[B*nW,win_size*win_size,C]
        x_windows=window_partition(shift_x.reshape(B,Hp*Wp,C),self.win_size)

        #W-MHSA/SW-MHSA
        attn_windows=self.window_attn(x_windows,mask)

        #merge_windows
        shift_x=window_recover(attn_windows,self.win_size,Hp,Wp).reshape(B,Hp,Wp,C)
        if self.shift_size>0:
            x=torch.roll(shift_x,(self.shift_size//2,self.shift_size//2),(1,2))
        else:
            x=shift_x

        if pad_r>0 or pad_l>0:
            x=x[:,:H,:W,:].contiguous()

        x=x.view(B,H*W,C)

        #FFN
        x=shortcut+self.drop_path(x)
        x=x+self.drop_path(self.mlp(self.norm2(x)))
        return x


In [38]:
class BasicLayer(nn.Module):
    def __init__(self, dim, depth, num_heads, window_size,
             mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0.,
             drop_path=0., norm_layer=nn.LayerNorm, downsample=None, use_checkpoint=False):
        super().__init__()
        self.dim = dim
        self.depth = depth
        self.window_size = window_size
        self.use_checkpoint = use_checkpoint
        self.shift_size = window_size // 2

        self.blocks=nn.ModuleList([
            Block(
                embed_dim=dim,
                head_num=num_heads,
                win_size=window_size,
                attn_drop=attn_drop,
                drop=drop,
                drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                qkv_bias=qkv_bias,
                mlp_ratio=int(mlp_ratio),
                shift_size=0 if i%2==0 else self.shift_size,
                norm_layer=norm_layer,
            )
            for i in range(depth)
        ])

        if downsample:
            self.downsample=downsample(dim=dim,norm_layer=norm_layer)
        else:
            self.downsample=None

    def forward(self,x,H,W):
        for block in self.blocks:
            block.H,block.W=H,W
            if not torch.jit.is_scripting() and self.use_checkpoint:
                x = checkpoint.checkpoint(block,x)
            else:
                x=block(x)
        if self.downsample is not None:
            x=self.downsample(x,H,W)
            H,W=(H+1)//2,(W+1)//2

        return x,H,W


In [39]:
class SwinTransformer(nn.Module):

    def __init__(self, patch_size=4, in_chans=3, num_classes=1000,
                 embed_dim=96, depths=(2, 2, 6, 2), num_heads=(3, 6, 12, 24),
                 window_size=7, mlp_ratio=4., qkv_bias=True,
                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1,
                 norm_layer=nn.LayerNorm, patch_norm=True,
                 use_checkpoint=False, **kwargs):
        super().__init__()

        self.num_classes = num_classes
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.patch_norm = patch_norm
        # stage4输出特征矩阵的channels
        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.mlp_ratio = mlp_ratio

        # split image into non-overlapping patches
        self.patch_embed = PatchEmbed(
            patch_size=patch_size, in_c=in_chans, embed_dim=embed_dim,
            norm_layer=norm_layer if self.patch_norm else None)
        self.pos_drop = nn.Dropout(p=drop_rate)

        # stochastic depth
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]  # stochastic depth decay rule

        # build layers
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            # 注意这里构建的stage和论文图中有些差异
            # 这里的stage不包含该stage的patch_merging层，包含的是下个stage的
            layers = BasicLayer(dim=int(embed_dim * 2 ** i_layer),
                                depth=depths[i_layer],
                                num_heads=num_heads[i_layer],
                                window_size=window_size,
                                mlp_ratio=self.mlp_ratio,
                                qkv_bias=qkv_bias,
                                drop=drop_rate,
                                attn_drop=attn_drop_rate,
                                drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                                norm_layer=norm_layer,
                                downsample=PatchMerging if (i_layer < self.num_layers - 1) else None,
                                use_checkpoint=use_checkpoint)
            self.layers.append(layers)

        self.norm = norm_layer(self.num_features)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(self.num_features, num_classes) if num_classes > 0 else nn.Identity()

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        # x: [B, L, C]
        x, H, W = self.patch_embed(x)
        x = self.pos_drop(x)

        for layer in self.layers:
            x, H, W = layer(x, H, W)

        x = self.norm(x)  # [B, L, C]
        x = self.avgpool(x.transpose(1, 2))  # [B, C, 1]
        x = torch.flatten(x, 1)
        x = self.head(x)
        return x


In [50]:
x=torch.rand((1,3,512,512))
def swin_tiny_patch4_window7_224(num_classes: int = 1000, **kwargs):
    # trained ImageNet-1K
    # https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth
    model = SwinTransformer(in_chans=3,
                            patch_size=4,
                            window_size=7,
                            embed_dim=96,
                            depths=(2, 2, 6, 2),
                            num_heads=(3, 6, 12, 24),
                            num_classes=num_classes,
                            **kwargs)
    return model
model=swin_tiny_patch4_window7_224()
model.cuda()
x=x.cuda()
y=model(x)
print(y.shape)

torch.Size([1, 1000])
